# 🚨 Modul 08: Deteksi Anomali (*Anomaly Detection*) & Dasar Text Mining
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/08_Anomaly_Detection_and_Text_Mining.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum menyentuh baris kode algoritma.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: ANOMALY & TEXT MINING]
```
Di modul-modul sebelumnya, kita fokus pada data numerik terstruktur yang rapi. Namun di industri modern:
1. **Deteksi Anomali**: Kita harus mampu menangkap penipuan perbankan (*Fraud Detection*), intrusi siber, dan mesin pabrik yang rusak sebelum meledak!
2. **Text Mining (NLP)**: Lebih dari 80% data dunia berbentuk **teks bebas tak terstruktur** (ulasan belanja, komentar media sosial, email komplain).

Mari kita taklukkan kedua domain esensial ini di **Modul 08**!


## 🛡️ Bagian 1: Deteksi Anomali dengan Isolation Forest

### 🏦 Analogi Dunia Nyata: Satpam Bank & Tiket Masuk Bioskop
Bayangkan seorang satpam bank memeriksa ribuan slip transaksi harian:
* **Transaksi Normal**: Bernilai wajar (Rp 50.000 s/d Rp 5.000.000) dan terjadi di jam kerja normal (08.00 - 17.00). Transaksi ini berkerumun padat seperti penonton bioskop di lobi.
* **Transaksi Anomali (*Outlier / Fraud*)**: Tiba-tiba ada transfer sebesar **Rp 950.000.000 pada pukul 03.15 subuh** dari lokasi antah berantah!

---

### 🌲 Bagaimana Algoritma Isolation Forest Bekerja?
Algoritma deteksi lama mencoba memetakan seluruh pola data normal terlebih dahulu (sangat lambat dan boros memori).  
**Isolation Forest (*iForest*)** menggunakan filosofi cerdas yang terbalik: **"Isolasi langsung si anomali!"**
1. Pohon acak memotong ruang data secara acak.
2. Titik data normal yang berkerumun padat membutuhkan **banyak potongan pohon (*Deep Path Length*)** untuk bisa terpisah sendirian.
3. Sebaliknya, titik anomali yang menyendiri jauh di sudut luar dapat **terisolasi hanya dengan 1 atau 2 potongan acak saja (*Short Path Length*)**!
4. **Parameter Kunci `contamination`**: Perkiraan persentase data pencilan/anomali dalam dataset (misal `0.05` atau 5%).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Simulasi Data Transaksi Keuangan: Nominal Transfer (Juta Rp) vs Durasi Transaksi (Detik)
np.random.seed(42)

# 1. Buat 300 transaksi normal (Berkerumun di nominal 2-10 juta, durasi 10-30 detik)
n_normal = 300
nominal_normal = np.random.normal(loc=5.0, scale=1.5, size=n_normal)
durasi_normal = np.random.normal(loc=20.0, scale=4.0, size=n_normal)

# 2. Buat 15 transaksi anomali mencurigakan (Nominal raksasa atau durasi transaksi ganjil)
n_anomali = 15
nominal_anomali = np.random.uniform(low=12.0, high=25.0, size=n_anomali)
durasi_anomali = np.random.uniform(low=2.0, high=50.0, size=n_anomali)

# Gabungkan ke dalam satu DataFrame
df_transaksi = pd.DataFrame({
    'Nominal_Transfer_Juta': np.concatenate([nominal_normal, nominal_anomali]),
    'Durasi_Detik': np.concatenate([durasi_normal, durasi_anomali])
})

print(f"Total Transaksi Simulasi: {len(df_transaksi)} baris")
df_transaksi.head()


In [ ]:
# Latih Model Isolation Forest
# contamination=0.05 artinya kita memperkirakan ~5% transaksi adalah anomali
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_transaksi['Status_Anomali'] = iso_forest.fit_predict(df_transaksi[['Nominal_Transfer_Juta', 'Durasi_Detik']])

# Output Isolation Forest: 1 = Normal, -1 = Anomali Mencurigakan!
df_transaksi['Label_Status'] = df_transaksi['Status_Anomali'].map({1: 'Normal', -1: 'Anomali/Fraud'})

jumlah_anomali = (df_transaksi['Status_Anomali'] == -1).sum()
print("=== HASIL DETEKSI ISOLATION FOREST ===")
print(f"✅ Transaksi Normal Terdeteksi : {(df_transaksi['Status_Anomali'] == 1).sum()} transaksi")
print(f"🚨 Transaksi Anomali / Fraud    : {jumlah_anomali} transaksi (Diberi bendera merah)")

# Visualisasi Scatter Plot Deteksi Anomali
plt.figure(figsize=(9, 5))
warna_map = {'Normal': 'royalblue', 'Anomali/Fraud': 'crimson'}

for label in ['Normal', 'Anomali/Fraud']:
    sub = df_transaksi[df_transaksi['Label_Status'] == label]
    plt.scatter(
        sub['Nominal_Transfer_Juta'],
        sub['Durasi_Detik'],
        c=warna_map[label],
        label=label,
        alpha=0.7 if label == 'Normal' else 0.9,
        s=40 if label == 'Normal' else 120,
        marker='o' if label == 'Normal' else 'X',
        edgecolor='k' if label == 'Anomali/Fraud' else 'none'
    )

plt.title("Deteksi Anomali Transaksi Perbankan via Isolation Forest", fontsize=12, fontweight='bold')
plt.xlabel("Nominal Transfer (Juta Rupiah)")
plt.ylabel("Durasi Transaksi (Detik)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

print("💡 Kesimpulan Praktisi:")
print("Titik merah 'X' otomatis diisolasi oleh sistem tanpa perlu aturan manual IF-ELSE yang kaku!")


## 📝 Bagian 2: Dasar Text Mining & Natural Language Processing (NLP)

### 🤖 Mengapa Komputer Butuh Text Mining?
Komputer adalah mesin kalkulator; komputer **tidak bisa membaca** tulisan teks seperti manusia. Komputer hanya memahami angka dan vektor matematis.  
Tugas *Text Mining* adalah: **Mengubah teks bahasa manusia menjadi representasi angka tanpa kehilangan makna kata (*Text Vectorization*)**.

---

### 🧹 4 Tahapan Standar Text Preprocessing:
1. **Case Folding**: Mengubah semua teks menjadi huruf kecil (*lowercase*), sehingga `"Bagus"`, `"BAGUS"`, dan `"bagus"` dihitung sebagai 1 kata yang sama.
2. **Punctuation & Noise Cleaning**: Menghapus tanda baca, angka, karakter aneh, dan spasi ganda menggunakan Regular Expression (`re`).
3. **Tokenizing**: Memecah kalimat panjang menjadi daftar kata-kata individu (*tokens*).
4. **Stopword Removal**: Membuang kata hubung umum yang sering muncul tapi minim informasi esensial (seperti: *"dan"*, *"yang"*, *"di"*, *"ke"*, *"dari"*, *"saya"*, *"ini"*).


## 🔢 3. Pembobotan Kata dengan TF-IDF (*Term Frequency - Inverse Document Frequency*)

Bagaimana cara memberi bobot nilai pada setiap kata? Kita menggunakan metode legendaris **TF-IDF**:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

* **TF (*Term Frequency*)**: Seberapa sering kata $t$ muncul di dalam ulasan tertentu $d$. Semakin sering muncul, semakin relevan kata tersebut bagi ulasan itu.
* **IDF (*Inverse Document Frequency*)**: Seberapa langka kata $t$ di seluruh kumpulan ulasan $D$:
  $$\text{IDF}(t) = \log \left( \frac{\text{Total Dokumen}}{\text{Jumlah Dokumen yang Memuat Kata } t} \right)$$
* **Intuisi Emas**:
  * Kata seperti *"produk"* muncul di hampir semua ulasan $\rightarrow$ Nilai IDF **rendah** (kurang informatif).
  * Kata seperti *"rusak"*, *"pecah"*, *"original"*, *"istimewa"* hanya muncul di ulasan tertentu $\rightarrow$ Nilai IDF **tinggi** (sangat membedakan sentimen positif vs negatif)!


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Dataset Riil Ulasan Belanja Online Bahasa Indonesia
data_ulasan = [
    # Ulasan Positif (Label = 1)
    ("Barang sangat bagus, kualitas original dan pengiriman super cepat!", 1),
    ("Pelayanan ramah, produk berfungsi dengan sangat baik, puas sekali!", 1),
    ("Bagus sekali sesuai ekspektasi, packaging rapi dan aman.", 1),
    ("Produk mantap, penjual responsif dan terpercaya, recommended seller!", 1),
    ("Kualitas premium, jahitan rapi dan bahan nyaman dipakai.", 1),
    ("Pengiriman kilat, barang original tanpa cacat sama sekali, bintang 5!", 1),
    
    # Ulasan Negatif (Label = 0)
    ("Barang jelek sekali, bahan tipis dan warna tidak sesuai foto!", 0),
    ("Sangat kecewa! Produk rusak saat sampai dan penjual tidak ramah.", 0),
    ("Pengiriman lambat sekali, packing asal-asalan dan barang cacat parah.", 0),
    ("Barang palsu, tidak bisa dipakai sama sekali, buang-buang uang!", 0),
    ("Respon penjual sangat buruk, barang tidak dikirim lengkap!", 0),
    ("Kecewa berat, kualitas sangat mengecewakan dan bau apek.", 0)
]

df_teks = pd.DataFrame(data_ulasan, columns=['Ulasan', 'Sentimen'])
print(f"Total Dataset Ulasan: {len(df_teks)} kalimat")
df_teks.head(4)


In [ ]:
# Stopwords sederhana Bahasa Indonesia
stop_words_id = [
    'dan', 'yang', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 
    'untuk', 'pada', 'adalah', 'saat', 'sekali', 'dan', 'sama'
]

def bersihkan_teks(teks):
    # 1. Lowercase
    teks = teks.lower()
    # 2. Hapus tanda baca dan angka
    teks = re.sub(r'[^a-z\s]', '', teks)
    # 3. Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df_teks['Ulasan_Clean'] = df_teks['Ulasan'].apply(bersihkan_teks)

print("Contoh Teks Sebelum vs Sesudah Cleaning:")
print(f"Asli   : {df_teks['Ulasan'].iloc[0]}")
print(f"Bersih : {df_teks['Ulasan_Clean'].iloc[0]}\n")

# Vektorisasi Teks menggunakan TF-IDF
tfidf = TfidfVectorizer(stop_words=stop_words_id)
X_tfidf = tfidf.fit_transform(df_teks['Ulasan_Clean'])

print(f"Dimensi Matriks TF-IDF: {X_tfidf.shape[0]} dokumen × {X_tfidf.shape[1]} kata unik")
print("Contoh 10 Kata Teratas dalam Kosakata (Vocabulary):")
print(tfidf.get_feature_names_out()[:10])


## 🎯 4. Klasifikasi Sentimen Ulasan dengan Naive Bayes

Sekarang, kita latih model **Multinomial Naive Bayes** pada representasi vektor TF-IDF di atas untuk membangun **Sistem Otomatis Deteksi Sentimen Pelanggan**!


In [ ]:
# Latih Classifier Multinomial Naive Bayes
model_sentimen = MultinomialNB()
model_sentimen.fit(X_tfidf, df_teks['Sentimen'])

# Uji Prediksi pada Data Latih
y_pred = model_sentimen.predict(X_tfidf)

print("=== EVALUASI MODEL KLASIFIKASI SENTIMEN ===")
print(classification_report(df_teks['Sentimen'], y_pred, target_names=['Negatif (0)', 'Positif (1)']))

# Visualisasi Confusion Matrix
cm = confusion_matrix(df_teks['Sentimen'], y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negatif (0)', 'Positif (1)'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax)
plt.title("Confusion Matrix Sentimen", fontweight='bold')
plt.grid(False)
plt.show()


## 🧪 5. Menguji Kalimat Ulasan Baru (*Live Sentiment Inference*)

Mari kita uji ketangguhan sistem klasifikasi sentimen kita dengan memberikan kalimat ulasan baru yang **sama sekali belum pernah dilihat sebelumnya oleh model**!


In [ ]:
# Kalimat uji baru dari pelanggan
ulasan_baru = [
    "Barang original mantap, pengiriman super cepat dan admin ramah!",
    "Parah sekali, barang rusak pas dibuka dan respon penjual sangat buruk!",
    "Bahan tebal dan jahitan rapi, saya sangat puas berbelanja di sini.",
    "Jangan beli di sini! Barangnya palsu dan pengiriman lambat sekali."
]

print("=== HASIL PREDIKSI SENTIMEN ULASAN BARU ===")
for u in ulasan_baru:
    u_clean = bersihkan_teks(u)
    u_vec = tfidf.transform([u_clean])
    pred = model_sentimen.predict(u_vec)[0]
    prob = model_sentimen.predict_proba(u_vec)[0]
    
    label_teks = "POSITIF ✅" if pred == 1 else "NEGATIF ❌"
    keyakinan = prob[pred] * 100
    
    print(f"Kalimat   : \"{u}\"")
    print(f"Prediksi  : {label_teks} (Keyakinan AI: {keyakinan:.1f}%)\n")


## 🧠 6. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep deteksi anomali dan text mining, lalu jalankan sel kode di bawah untuk memeriksa pembahasannya:

---
* **Soal 1**: Mengapa algoritma Isolation Forest jauh lebih cepat mendeteksi anomali dibandingkan algoritma klasterisasi tradisional?
* **Soal 2**: Apa makna dari parameter `contamination` pada Isolation Forest?
* **Soal 3**: Mengapa proses *Stopword Removal* penting dalam prapemrosesan teks?
* **Soal 4**: Mengapa kata yang muncul di seluruh dokumen teks mendapatkan nilai bobot TF-IDF yang sangat rendah?


In [ ]:
# Kunci jawaban evaluasi mandiri Modul 8
kunci_jawaban_modul_8 = {
    "Soal 1": "Karena Isolation Forest langsung mengisolasi titik anomali menggunakan pemisahan acak. Data anomali berada di tepi luar sehingga membutuhkan path pemotongan pohon yang jauh lebih pendek dibanding data normal.",
    "Soal 2": "Parameter 'contamination' adalah estimasi proporsi/persentase titik anomali yang diharapkan berada dalam dataset (misal 0.05 berarti 5% dari total data adalah potensi pencilan/anomali).",
    "Soal 3": "Stopword removal membuang kata sambung umum yang tidak membawa informasi pembeda sentimen/topik, sehingga mengurangi dimensi kosakata dan menghemat memori komputasi.",
    "Soal 4": "Karena nilai IDF (Inverse Document Frequency) menghitung kebalikan dari frekuensi dokumen. Jika suatu kata muncul di semua dokumen, nilai log(N/N) mendekati 0 sehingga bobot informasinya diredam."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 08 ===")
for soal, pembahasan in kunci_jawaban_modul_8.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 7. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Analisis Sentimen Ulasan Anda Sendiri!
Cobalah masukkan 1 kalimat ulasan Anda sendiri ke dalam kotak di bawah ini, bersihkan teksnya, dan biarkan model AI memprediksi apakah kalimat Anda berbau pujian (Positif) atau komplain (Negatif)!


In [ ]:
# Masukkan kalimat buatan Anda di bawah ini:
kalimat_anda = "Paket sudah mendarat dengan selamat, packing super tebal dan produk asli mantap!"

# Proses inferensi otomatis
kalimat_clean = bersihkan_teks(kalimat_anda)
kalimat_vec = tfidf.transform([kalimat_clean])
pred_anda = model_sentimen.predict(kalimat_vec)[0]
prob_anda = model_sentimen.predict_proba(kalimat_vec)[0]

status = "POSITIF 🌟" if pred_anda == 1 else "NEGATIF ⚠️"
print(f"Ulasan Anda : \"{kalimat_anda}\"")
print(f"Hasil AI    : Sentimen {status} (Skor Keyakinan: {prob_anda[pred_anda]*100:.2f}%)")


## 🎯 8. Checklist Kelulusan Modul 08 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami konsep deteksi anomali (*Fraud Detection*) dan filosofi *short path length* pada **Isolation Forest**.
- [ ] Saya memahami fungsi parameter `contamination` dan cara membaca label status ($1$ = Normal, $-1$ = Anomali).
- [ ] Saya menguasai 4 tahapan *Text Preprocessing*: Lowercasing, Punctuation Cleaning, Tokenizing, dan Stopword Removal.
- [ ] Saya memahami intuisi pembobotan **TF-IDF** (mengapa kata langka bernilai bobot lebih tinggi).
- [ ] Saya mampu melatih model **Naive Bayes** untuk klasifikasi sentimen ulasan teks bahasa Indonesia.
- [ ] Saya berhasil melakukan pengujian (*inference*) pada kalimat ulasan baru.

---
🚀 **Luar Biasa! Anda telah menaklukkan Modul 08!**  
Di modul berikutnya (**Modul 09**), kita akan melangkah ke level profesional rekayasa machine learning: **Hyperparameter Tuning (GridSearchCV), Scikit-Learn Pipeline, dan Serialisasi Model (`joblib`)** agar model kita siap dipasang di aplikasi nyata!
